# CTIG — Cultural Text-to-Image Generation (Vietnam)

Chạy pipeline v1 trên Kaggle GPU. **Settings:** Accelerator = GPU T4 x2 (hoặc T4), Internet = On.

Sửa `REPO_URL` ở cell dưới thành repo của bạn.

In [ ]:
REPO_URL = "https://github.com/OxyzGiaHuy/CTIG.git"   # <- sửa
CONFIG   = "configs/kaggle_t4x2.yaml"             # 1 GPU: configs/kaggle_t4.yaml

import os, subprocess
if not os.path.exists("/kaggle/working/CTIG"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/CTIG"], check=True)
%cd /kaggle/working/CTIG
!git pull -q
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt
import transformers, diffusers; print("transformers", transformers.__version__, "| diffusers", diffusers.__version__)

## Smoke test: 2 prompt
Lần đầu tải model ~10 phút. Nếu chạy xong và có ảnh là toàn bộ các khối đã hoạt động.

In [ ]:
!python -m ctig.cli batch --config $CONFIG --ids p001,p050 --run-name smoke

In [ ]:
# Xem ảnh cuối và các vòng của từng prompt
import json
from pathlib import Path
from IPython.display import display, Image, Markdown

def show(run_name, max_prompts=10):
    run = Path("/kaggle/working/runs") / run_name
    recs = json.loads((run / "records.json").read_text(encoding="utf-8"))
    for r in recs[:max_prompts]:
        display(Markdown(f"### {r['prompt_id']} — {r['prompt_text']}  \n"
                         f"đạt **{r['passed']}** · CLIP {r['clip_fidelity']:.2f} · judge {r['judge_score']:.2f} · {r['iterations']} vòng"))
        for i, p in enumerate(r["iteration_images"]):
            display(Markdown(f"vòng {i}"))
            display(Image(filename=p, width=384))
        if r["residual_findings"]:
            display(Markdown("còn sót: " + "; ".join(r["residual_findings"][:3])))

show("smoke")

## Chạy đủ 50 prompt
~3 giờ trên 2×T4, ~6 giờ trên 1×T4. Kết quả được ghi sau **mỗi** prompt, phiên ngắt vẫn giữ được phần đã chạy. Bỏ `--limit` để chạy hết.

In [ ]:
!python -m ctig.cli batch --config $CONFIG --run-name v1-full --limit 50

In [ ]:
import json; print(json.dumps(json.load(open("/kaggle/working/runs/v1-full/summary.json")), indent=1, ensure_ascii=False)[:1500])
show("v1-full", max_prompts=6)

## Tải kết quả về
`report.html` mở bằng trình duyệt ở máy bạn (ảnh dùng đường dẫn tương đối trong thư mục run).

In [ ]:
!cd /kaggle/working && zip -qr ctig_runs.zip runs -x "runs/_cache/*" && ls -lh ctig_runs.zip